# 02 · Lightweight 1D CNN — Axis × Position × LOSO

**Project:** Sensor-Efficient Wearable Gym Exercise Recognition: An Axis-Level and Sensor-Placement Analysis Under Subject-Independent Evaluation

Runs one **small, fixed-architecture 1D CNN** on the same 15 axis configurations × 3 positions × 10 LOSO folds as notebook 01, on the **identical windows**. All numbers come from the real file on your Colab/Kaggle run; nothing is reported until you run it.

### Design decisions frozen **before** any model was run
| Decision | Value | Reason |
|---|---|---|
| Windows | Same as 01: 4 s = 80 samples @ 20 Hz, 50 % overlap, never crossing a (subject, position, session, label) boundary | Same inputs → CNN vs classical differences come from the model, not the data. The notebook **rebuilds** the windows and asserts they match `windows_index` saved by 01. |
| Input | Raw channels only (`k` channels × 80 samples). No hand-crafted features, no cross-channel features | A k-axis configuration sees exactly k input channels |
| Normalisation | Per-channel z-score, mean/std computed on **training-subject windows of each fold only** | No test-subject influence |
| Architecture | Conv(k→32,5)-BN-ReLU-Pool → Conv(32→64,5)-BN-ReLU-Pool → Conv(64→128,3)-BN-ReLU → global avg pool → Dropout 0.3 → Linear(128→12). ≈ 38 k parameters | Small enough to be a credible wearable model. **Same architecture for every configuration**; only the first conv's input width changes |
| Training | AdamW, one-cycle LR (max 3e-3), weight decay 1e-3, batch 256, **fixed 25 epochs**, weighted cross-entropy | **No early stopping and no model selection**, because there is no validation signal that could involve the test subject |
| Class imbalance | Class weights from the training windows only (same formula as 01) | `Null` is an extreme class |
| Hyper-parameters | **Fixed, pre-declared, not tuned** (same policy as 01) | Keeps the classical-vs-CNN comparison symmetric. Nested subject-grouped tuning, if you want it, is a separate and clearly labelled step |
| Metrics | Identical to 01 (macro-F1 over classes present in the held-out subject; macro-F1 over the 11 workouts; accuracy; weighted-F1; macro precision/recall; per-class F1; long-format confusion matrices) | Result files concatenate directly in notebook 03 |
| Efficiency | Channels, input data rate (channels × 20 Hz), exact parameter count, MACs per window, serialized size, single-window **CPU** latency (batch 1, 1 thread) | Hardware is recorded in `environment.json`; this is **not** edge-device latency |

### Leakage guards implemented
- Folds are by **subject**; assertions in every fold that the test subject is absent from the training windows and that the training subjects equal the fold definition.
- Normalisation statistics, class weights and training use training windows only; a unit test recomputes the statistics independently with NumPy.
- **Label-shuffle control:** a model trained on shuffled training labels must score near chance on the held-out subject (asserted). A leak would show up as a score far above chance.
- Windows are verified identical to the ones notebook 01 used.
- Caveat for the paper (unchanged from 01): the dataset was already normalised by its providers with an undocumented method, which this pipeline cannot undo.

### Resumable
Each (position, fold, config, seed) unit is appended to `results.csv` right after it finishes. If Colab disconnects, re-run everything: finished units are skipped. A config-hash check refuses to mix results made with different settings. Adding seeds later is allowed (the seed is part of the unit key and not of the hash).

> **First run with `SMOKE_TEST = True`** (default) to verify everything and read the timing/ETA, then set `SMOKE_TEST = False` for the full grid. Smoke results go in a separate folder and are never mixed with the full run. **Use a GPU runtime** (Runtime → Change runtime type → T4) for the full run.

## 1 · Configuration

In [ ]:
import os, sys, re, json, glob, time, hashlib, platform, random, shutil, zipfile, warnings, itertools, io, copy
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ============================ USER CONFIG ====================================
SMOKE_TEST = bool(os.environ.get("SMOKE_TEST", "1") == "1")   # True = tiny check run; False = full grid
SEED = 42
SEEDS = [SEED]               # training seeds per unit. Add e.g. [42, 43, 44] later to re-run with more seeds (existing units are kept)
DATA_PATH = os.environ.get("RECGYM_CSV", "data/RecGym.csv")
RESULTS_ROOT = os.environ.get("RESULTS_DIR", "results/raw/cnn")   # parent dir; "full" or "smoke" is appended below
CLASSICAL_ROOT = os.environ.get("CLASSICAL_DIR", "results/raw/classical")   # notebook 01 output, for the descriptive comparison
EXPECTED_SHA256 = "2031c077a57e93d6b5c39da198470cb6c9b1a3c0652ab57cfc5d6a39bfa840e9"   # same file as notebooks 00/01
ENFORCE_HASH = os.environ.get("RECGYM_SKIP_HASH") is None
MOUNT_DRIVE = True
# (RESULTS_ROOT and CLASSICAL_ROOT are set from environment variables above.)

FS = 20
WINDOW_SECONDS = 4
OVERLAP = 0.5                # MUST equal notebook 01 (checked against its saved window index)
POSITIONS = None             # None = all; or e.g. ["wrist"]
CONFIG_NAMES = None          # None = all 15; or e.g. ["6-axis", "acc-3"]
FOLDS = None                 # None = all LOSO folds; or list of fold indices

CNN_PARAMS = dict(widths=(32, 64, 128), kernels=(5, 5, 3), dropout=0.3)
TRAIN_PARAMS = dict(epochs=25, batch_size=256, max_lr=3e-3, weight_decay=1e-3)
LATENCY_N_WINDOWS = 50
ALLOW_CPU = False            # the full grid on CPU would take very long; set True only on purpose
FORCE_NEW_RUN = False        # True = ignore a config-hash mismatch and overwrite run_config.json (results.csv is NOT deleted)
# ==============================================================================

if SMOKE_TEST:
    CONFIG_NAMES = CONFIG_NAMES or ["6-axis", "single-Ax"]
    FOLDS = FOLDS or [0, 1]
    TRAIN_PARAMS = {**TRAIN_PARAMS, "epochs": 3}

IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE = Path("/kaggle/working") if IN_KAGGLE else (Path("/content") if IN_COLAB else Path(".").resolve())
random.seed(SEED); np.random.seed(SEED)

if IN_COLAB and MOUNT_DRIVE and str(DATA_PATH).startswith("/content/drive") and not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

RESULTS_ROOT = Path(RESULTS_ROOT)
CLASSICAL_ROOT = Path(CLASSICAL_ROOT) if CLASSICAL_ROOT else RESULTS_ROOT.parent / "classical"
RUN_DIR = RESULTS_ROOT / ("smoke" if SMOKE_TEST else "full")
RUN_DIR.mkdir(parents=True, exist_ok=True)

LOG = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)

import torch
import torch.nn as nn
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu" and not SMOKE_TEST and not ALLOW_CPU:
    raise RuntimeError("No GPU visible. Switch the runtime to a GPU (Colab: Runtime > Change runtime type > T4) "
                       "or set ALLOW_CPU = True if you really want the full grid on CPU.")
torch.backends.cudnn.benchmark = True      # speed; GPU kernels are then not bit-reproducible (recorded in environment.json)

log("SMOKE_TEST =", SMOKE_TEST, "| device:", DEVICE, "| results dir:", RUN_DIR)
if DEVICE.type == "cuda": log("GPU:", torch.cuda.get_device_name(0))
if SMOKE_TEST:
    log(">>> SMOKE TEST: reduced grid and only", TRAIN_PARAMS["epochs"], "epochs. Set SMOKE_TEST = False for the real run. <<<")

## 2 · Load data and verify it is the audited file

In [ ]:
def sha256_of(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

assert os.path.exists(DATA_PATH), f"Data file not found: {DATA_PATH}. Mount Drive or set DATA_PATH."
DATA_SHA256 = sha256_of(DATA_PATH)
log("SHA-256:", DATA_SHA256)
if ENFORCE_HASH:
    assert DATA_SHA256 == EXPECTED_SHA256, ("The file differs from the one audited in notebook 00. "
        "Re-run the audit on this file (or update EXPECTED_SHA256 on purpose).")
    log("Hash matches the audited file.")

raw = pd.read_csv(DATA_PATH)
raw.columns = [str(c).strip() for c in raw.columns]
log("Loaded:", raw.shape, list(raw.columns))

def norm(c): return re.sub(r"[^a-z0-9]", "", str(c).lower())
ncol = {norm(c): c for c in raw.columns}
def pick(cands):
    for k in cands:
        if k in ncol: return ncol[k]
    return None
subj_col = pick(["subject", "object", "subjectid", "volunteer", "participant", "user"])
label_col = pick(["workout", "label", "activity", "exercise"])
pos_col = pick(["position", "placement", "location"])
sess_col = pick(["session", "sessionid", "day"])
IMU = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
chan_cols = {k: ncol.get(k.lower()[0] + "_" + k.lower()[1]) or ncol.get(k.lower()) for k in IMU}
assert subj_col and label_col and pos_col, f"Cannot detect roles in {list(raw.columns)}"
assert all(chan_cols.values()), f"Cannot detect IMU channels: {chan_cols}"
log("Roles:", dict(subject=subj_col, label=label_col, position=pos_col, session=sess_col), "| channels:", chan_cols)
if sess_col is None:
    log("WARNING: no session column; runs are delimited by (subject, position, label) only.")

subject_str = raw[subj_col].astype(str).str.strip()
subjects = sorted(subject_str.unique(), key=lambda s: (len(s), s))
labels = sorted(raw[label_col].astype(str).str.strip().unique())
positions_all = sorted(raw[pos_col].astype(str).str.strip().unique())
K = len(labels)
label_map = {l: i for i, l in enumerate(labels)}
null_ids = [label_map[l] for l in labels if "null" in l.lower()]
log("Subjects:", subjects); log("Positions:", positions_all); log("Labels:", labels, "| null ids:", null_ids)

cat = lambda s, cats: pd.Categorical(s.astype(str).str.strip(), categories=cats).codes.astype(np.int32)
subj_c = cat(raw[subj_col], subjects)
pos_c = cat(raw[pos_col], positions_all)
lab_c = cat(raw[label_col], labels)
sess_vals = sorted(raw[sess_col].astype(str).str.strip().unique()) if sess_col else ["NA"]
sess_c = cat(raw[sess_col], sess_vals) if sess_col else np.zeros(len(raw), dtype=np.int32)
assert (subj_c >= 0).all() and (pos_c >= 0).all() and (lab_c >= 0).all()
sig = raw[[chan_cols[k] for k in IMU]].to_numpy(dtype=np.float32)
assert np.isfinite(sig).all(), "non-finite IMU values: handle explicitly before modelling"
n_rows = len(raw); del raw
log("Sessions:", sess_vals)

## 3 · Rebuild the windows and verify they are identical to notebook 01's
Same code as notebook 01. If 01's saved window index is found, every window start, label, subject and position is compared; any difference stops the run.

In [ ]:
W = int(round(WINDOW_SECONDS * FS)); S = max(1, int(round(W * (1 - OVERLAP))))
chg = np.ones(n_rows, dtype=bool)
chg[1:] = ((subj_c[1:] != subj_c[:-1]) | (pos_c[1:] != pos_c[:-1]) |
           (sess_c[1:] != sess_c[:-1]) | (lab_c[1:] != lab_c[:-1]))
starts = np.flatnonzero(chg); lens = np.diff(np.r_[starts, n_rows])
log(f"Runs: {len(starts)} | window = {W} samples, step = {S}")
keep = lens >= W
rs, rl = starts[keep], lens[keep]
nwin = (rl - W) // S + 1
tot = int(nwin.sum())
run_of_win = np.repeat(np.arange(len(rs)), nwin)
offs = np.arange(tot) - np.repeat(np.cumsum(nwin) - nwin, nwin)
w_start = (rs[run_of_win] + offs * S).astype(np.int64)
w_end = w_start + W - 1
for nm, arr in [("subject", subj_c), ("position", pos_c), ("session", sess_c), ("label", lab_c)]:
    assert (arr[w_start] == arr[w_end]).all(), f"window crosses a {nm} boundary"
widx = pd.DataFrame({"start_row": w_start, "subject": [subjects[i] for i in subj_c[w_start]],
                     "position": [positions_all[i] for i in pos_c[w_start]],
                     "label": [labels[i] for i in lab_c[w_start]], "label_id": lab_c[w_start].astype(np.int16)})
WINDOWS_SHA256 = hashlib.sha256(w_start.tobytes()).hexdigest()
log("Windows:", len(widx), "| window-start hash:", WINDOWS_SHA256[:16], "...")

# ---- verify against the index saved by notebook 01
ref = None
p_parq, p_csv = CLASSICAL_ROOT / "shared" / "windows_index.parquet", CLASSICAL_ROOT / "shared" / "windows_index.csv.gz"
try:
    if p_parq.exists(): ref = pd.read_parquet(p_parq)
    elif p_csv.exists(): ref = pd.read_csv(p_csv)
except Exception as e:
    log("Could not read notebook 01's window index:", e)
if ref is not None:
    assert len(ref) == len(widx), f"window count differs from notebook 01: {len(ref)} vs {len(widx)}"
    assert (ref["start_row"].to_numpy() == w_start).all(), "window starts differ from notebook 01"
    assert (ref["label_id"].to_numpy() == widx["label_id"].to_numpy()).all(), "window labels differ from notebook 01"
    for c in ["subject", "position"]:
        assert (ref[c].astype(str).to_numpy() == widx[c].astype(str).to_numpy()).all(), f"window {c} differs from notebook 01"
    mp = CLASSICAL_ROOT / "shared" / "windows_meta.json"
    if mp.exists():
        meta = json.load(open(mp))
        assert meta["window_samples"] == W and meta["step"] == S and meta["labels"] == labels and meta["data_sha256"] == DATA_SHA256, \
            "windows_meta.json from notebook 01 does not match this run's settings"
    log("VERIFIED: windows are identical to notebook 01's.")
else:
    log("WARNING: notebook 01's window index was not found at", CLASSICAL_ROOT / "shared",
        "-> windows are rebuilt with the same code but could NOT be compared. Set CLASSICAL_ROOT if the files live elsewhere.")
del ref

def gather(rows):                       # raw windows for given start rows -> (n, W, 6)
    return sig[rows[:, None] + np.arange(W)[None, :]]

POS_LIST = POSITIONS or positions_all
POS_DATA = {}
for p in POS_LIST:
    sel = np.flatnonzero(widx["position"].to_numpy() == p)
    Xp = gather(w_start[sel])                                      # (n, W, 6)
    POS_DATA[p] = dict(X=np.ascontiguousarray(Xp.transpose(0, 2, 1)),   # (n, 6, W) for Conv1d
                       y=widx["label_id"].to_numpy()[sel].astype(np.int64),
                       subj=widx["subject"].to_numpy()[sel], starts=w_start[sel])
    log(f"{p}: windows={len(sel)}, tensor={POS_DATA[p]['X'].shape}")
    del Xp

## 4 · Axis configurations, folds and the model

In [ ]:
acc, gyr = ["Ax", "Ay", "Az"], ["Gx", "Gy", "Gz"]
ALL_CONFIGS = {"6-axis": IMU, "acc-3": acc, "gyro-3": gyr}
for a, b in itertools.combinations(acc, 2): ALL_CONFIGS[f"acc-{a}{b}"] = [a, b]
for a, b in itertools.combinations(gyr, 2): ALL_CONFIGS[f"gyro-{a}{b}"] = [a, b]
for k in IMU: ALL_CONFIGS[f"single-{k}"] = [k]
CONFIGS = {k: v for k, v in ALL_CONFIGS.items() if (CONFIG_NAMES is None or k in CONFIG_NAMES)}
FOLD_DEFS = [{"fold": i, "test_subject": s, "train_subjects": [x for x in subjects if x != s]} for i, s in enumerate(subjects)]
FOLD_LIST = [f for f in FOLD_DEFS if (FOLDS is None or f["fold"] in FOLDS)]
assert len(ALL_CONFIGS) == 15
log("Configs:", list(CONFIGS), "| folds:", [f["fold"] for f in FOLD_LIST], "| positions:", POS_LIST, "| seeds:", SEEDS)

from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix

class CNN1D(nn.Module):
    def __init__(self, in_ch, n_classes, widths=(32, 64, 128), kernels=(5, 5, 3), dropout=0.3):
        super().__init__()
        layers, c = [], in_ch
        for i, (w, ks) in enumerate(zip(widths, kernels)):
            layers += [nn.Conv1d(c, w, ks, padding=ks // 2, bias=False), nn.BatchNorm1d(w), nn.ReLU()]
            if i < len(widths) - 1: layers.append(nn.MaxPool1d(2))
            c = w
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(c, n_classes)
    def forward(self, x):                      # x: (B, in_ch, T)
        return self.fc(self.drop(self.pool(self.features(x)).squeeze(-1)))

def count_params(m): return int(sum(p.numel() for p in m.parameters()))

def count_macs(m, in_ch, T=W):
    tot = [0]
    def hook(mod, inp, out):
        if isinstance(mod, nn.Conv1d): tot[0] += out.numel() * (mod.in_channels // mod.groups) * mod.kernel_size[0]
        elif isinstance(mod, nn.Linear): tot[0] += mod.in_features * mod.out_features
    hs = [mod.register_forward_hook(hook) for mod in m.modules() if isinstance(mod, (nn.Conv1d, nn.Linear))]
    was = m.training; m.eval()
    with torch.no_grad(): m(torch.zeros(1, in_ch, T, device=next(m.parameters()).device))
    m.train(was)
    for h in hs: h.remove()
    return int(tot[0])

def serialized_size(m):
    buf = io.BytesIO(); torch.save(m.state_dict(), buf); return buf.tell()

def balanced_weights(y, K):
    cnt = np.bincount(y, minlength=K).astype(float)
    return np.where(cnt > 0, len(y) / (K * np.maximum(cnt, 1)), 0.0)

def train_cnn(Xtr, ytr, n_ch, seed, epochs, shuffle_labels=False):
    # Xtr: (N, n_ch, W) float32 tensor on DEVICE (already normalised with training-fold statistics); ytr: int64 numpy
    torch.manual_seed(seed); np.random.seed(seed)
    if shuffle_labels:                                    # used ONLY by the label-shuffle control
        ytr = np.random.RandomState(seed).permutation(ytr)
    model = CNN1D(n_ch, K, **CNN_PARAMS).to(DEVICE)
    cw = torch.tensor(balanced_weights(ytr, K), dtype=torch.float32, device=DEVICE)
    yt = torch.from_numpy(ytr).to(DEVICE)
    bs = TRAIN_PARAMS["batch_size"]; N = len(yt); steps = max(1, N // bs)      # drop the last partial batch (BatchNorm-safe)
    opt = torch.optim.AdamW(model.parameters(), lr=TRAIN_PARAMS["max_lr"], weight_decay=TRAIN_PARAMS["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=TRAIN_PARAMS["max_lr"], total_steps=epochs * steps)
    lossf = nn.CrossEntropyLoss(weight=cw)
    g = torch.Generator(device=DEVICE.type); g.manual_seed(seed)
    last = float("nan")
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(N, device=DEVICE, generator=g)
        run = torch.zeros((), device=DEVICE)
        for i in range(steps):
            idx = perm[i * bs:(i + 1) * bs]
            opt.zero_grad(set_to_none=True)
            loss = lossf(model(Xtr[idx]), yt[idx])
            loss.backward(); opt.step(); sched.step()
            run += loss.detach()
        last = float(run / steps)
    return model, last

@torch.no_grad()
def predict_cnn(model, X, bs=4096):
    model.eval()
    return torch.cat([model(X[i:i + bs]).argmax(1) for i in range(0, len(X), bs)]).cpu().numpy()

def prepare_fold(D, Xp_t, test_s):
    # Returns normalised train/test tensors for one fold. Statistics come from the TRAINING subjects only.
    te = D["subj"] == test_s; tr = ~te
    assert test_s not in set(D["subj"][tr]), "test subject present in training windows"
    tr_i = torch.from_numpy(np.flatnonzero(tr)).to(DEVICE); te_i = torch.from_numpy(np.flatnonzero(te)).to(DEVICE)
    Xtr_raw = Xp_t[tr_i]
    mu = Xtr_raw.mean(dim=(0, 2), keepdim=True); sd = Xtr_raw.std(dim=(0, 2), keepdim=True).clamp_min(1e-6)
    Xtr = (Xtr_raw - mu) / sd; del Xtr_raw
    Xte = (Xp_t[te_i] - mu) / sd
    return dict(tr=tr, te=te, Xtr=Xtr, Xte=Xte, mu=mu.flatten().cpu().numpy(), sd=sd.flatten().cpu().numpy())

def measure_latency(model, mu, sd, ch_idx, te_rows, n_warm=5):
    # single window, batch 1, 1 CPU thread: slice raw samples -> z-score -> forward -> argmax
    m = copy.deepcopy(model).to("cpu").eval()
    prev = torch.get_num_threads(); torch.set_num_threads(1)
    muc, sdc = mu[ch_idx][:, None], sd[ch_idx][:, None]
    lf, lp = [], []
    with torch.inference_mode():
        x0 = torch.from_numpy(((sig[te_rows[0]:te_rows[0] + W][:, ch_idx].T - muc) / sdc).astype(np.float32))[None]
        for _ in range(n_warm): m(x0)
        for r0 in te_rows:
            a = time.perf_counter()
            x = torch.from_numpy(((sig[r0:r0 + W][:, ch_idx].T - muc) / sdc).astype(np.float32))[None]
            b = time.perf_counter()
            _ = int(m(x).argmax(1)); c = time.perf_counter()
            lf.append(b - a); lp.append(c - b)
    torch.set_num_threads(prev)
    return 1000 * float(np.median(lf)), 1000 * float(np.median(lp))

_m = CNN1D(6, K, **CNN_PARAMS)
log("Model (6 channels):", count_params(_m), "parameters,", count_macs(_m, 6), "MACs/window,", serialized_size(_m) // 1024, "KB")
for _c in [1, 2, 3]:
    _mm = CNN1D(_c, K, **CNN_PARAMS); log(f"   {_c} channel(s): {count_params(_mm)} parameters, {count_macs(_mm, _c)} MACs/window")
del _m

## 5 · Run integrity (config hash)

In [ ]:
design = {"window_samples": W, "step": S, "fs": FS, "labels": labels, "positions": POS_LIST, "subjects": subjects,
          "cnn": CNN_PARAMS, "train": TRAIN_PARAMS, "data_sha256": DATA_SHA256, "windows_sha256": WINDOWS_SHA256,
          "smoke": SMOKE_TEST, "seed_base": SEED}          # SEEDS deliberately excluded: the seed is part of the unit key
cfg_hash = hashlib.sha256(json.dumps(design, sort_keys=True, default=str).encode()).hexdigest()
cfg_path = RUN_DIR / "run_config.json"
if cfg_path.exists():
    old = json.load(open(cfg_path))
    if old.get("hash") != cfg_hash and not FORCE_NEW_RUN:
        raise RuntimeError("Existing results were produced with DIFFERENT settings. Use a new RESULTS_ROOT, or set FORCE_NEW_RUN=True "
                           "after deleting results.csv / confusion_long.csv yourself.")
json.dump({"hash": cfg_hash, "design": design, "seeds": SEEDS}, open(cfg_path, "w"), indent=2, default=str)
RES_PATH, CONF_PATH = RUN_DIR / "results.csv", RUN_DIR / "confusion_long.csv"
done = set()
if RES_PATH.exists():
    _d = pd.read_csv(RES_PATH, usecols=["position", "config", "model", "fold", "seed"])
    done = set(map(tuple, _d.to_numpy().tolist()))
log("Units already completed:", len(done))

## 6 · Sanity checks before the grid
1. The fold normalisation statistics equal an independent NumPy computation on the training subjects only.
2. **Label-shuffle control:** training on shuffled labels must give near-chance macro-F1 on the held-out subject. If it does not, something leaks and the run stops.

In [ ]:
D0 = POS_DATA[POS_LIST[0]]
fd0 = FOLD_LIST[0]
Xp0 = torch.from_numpy(D0["X"]).to(DEVICE)
pf = prepare_fold(D0, Xp0, fd0["test_subject"])
tr0 = D0["X"][pf["tr"]].astype(np.float64)
assert np.allclose(pf["mu"], tr0.mean(axis=(0, 2)), atol=1e-4), "normalisation mean is not the training-subject mean"
assert np.allclose(pf["sd"], tr0.std(axis=(0, 2), ddof=1), atol=1e-3), "normalisation std is not the training-subject std"
del tr0
log("Check 1 passed: normalisation statistics come from training subjects only.")

t0 = time.time()
m_sh, _ = train_cnn(pf["Xtr"], D0["y"][pf["tr"]], 6, SEED, epochs=min(TRAIN_PARAMS["epochs"], 6), shuffle_labels=True)
pred_sh = predict_cnn(m_sh, pf["Xte"]); y_sh = D0["y"][pf["te"]]
f1_shuffled = float(f1_score(y_sh, pred_sh, labels=np.unique(y_sh), average="macro", zero_division=0))
chance_ref = 1.0 / K
log(f"Check 2: label-shuffle control on {POS_LIST[0]} fold {fd0['fold']}: macro-F1 = {f1_shuffled:.3f} "
    f"(chance ~ {chance_ref:.3f}) in {time.time() - t0:.1f}s")
assert f1_shuffled < 0.25, "shuffled-label model is far above chance -> LEAKAGE or a bug. Do not trust any result."
json.dump({"label_shuffle_macro_f1": f1_shuffled, "chance_reference": chance_ref, "position": POS_LIST[0], "fold": fd0["fold"],
           "threshold": 0.25}, open(RUN_DIR / "sanity_checks.json", "w"), indent=2)
del m_sh, pf, Xp0
if DEVICE.type == "cuda": torch.cuda.empty_cache()

## 7 · Main LOSO loop (resumable)

In [ ]:
workout_ids = [i for i in range(K) if i not in null_ids]
total_units = len(POS_LIST) * len(FOLD_LIST) * len(CONFIGS) * len(SEEDS)
unit_i, t_start, n_new = 0, time.time(), 0
rows_buf, conf_buf = [], []

def flush():
    global rows_buf, conf_buf
    if rows_buf:
        pd.DataFrame(rows_buf).to_csv(RES_PATH, mode="a", header=not RES_PATH.exists(), index=False); rows_buf = []
    if conf_buf:
        pd.DataFrame(conf_buf).to_csv(CONF_PATH, mode="a", header=not CONF_PATH.exists(), index=False); conf_buf = []

for p in POS_LIST:
    D = POS_DATA[p]
    Xp_t = torch.from_numpy(D["X"]).to(DEVICE)
    for fd in FOLD_LIST:
        test_s = fd["test_subject"]
        pending = [(c, s) for c in CONFIGS for s in SEEDS if (p, c, "CNN", fd["fold"], s) not in done]
        if not pending:
            continue
        pf = prepare_fold(D, Xp_t, test_s); tr, te = pf["tr"], pf["te"]
        assert set(D["subj"][tr]) == set(fd["train_subjects"]), "training subjects differ from fold definition"
        ytr, yte = D["y"][tr], D["y"][te]
        assert len(np.unique(ytr)) == K, "a class is missing from the training subjects"
        present = np.unique(yte)
        te_rows = D["starts"][te][:LATENCY_N_WINDOWS]
        for cname, ch in CONFIGS.items():
            ch_idx = [IMU.index(c) for c in ch]; ch_t = torch.tensor(ch_idx, device=DEVICE)
            Xtr_c = pf["Xtr"].index_select(1, ch_t).contiguous(); Xte_c = pf["Xte"].index_select(1, ch_t).contiguous()
            assert Xtr_c.shape[1] == len(ch) and Xte_c.shape[1] == len(ch)
            for seed in SEEDS:
                unit_i += 1
                key = (p, cname, "CNN", fd["fold"], seed)
                if key in done: continue
                if DEVICE.type == "cuda": torch.cuda.synchronize()
                t0 = time.time()
                model, last_loss = train_cnn(Xtr_c, ytr, len(ch), seed + fd["fold"], TRAIN_PARAMS["epochs"])
                if DEVICE.type == "cuda": torch.cuda.synchronize()
                t_fit = time.time() - t0
                t0 = time.time(); pred = predict_cnn(model, Xte_c); t_pred = time.time() - t0
                cm = confusion_matrix(yte, pred, labels=np.arange(K))
                f1c = f1_score(yte, pred, labels=np.arange(K), average=None, zero_division=0)
                pw = [i for i in workout_ids if i in present]
                row = dict(position=p, config=cname, model="CNN", fold=fd["fold"], seed=seed, test_subject=test_s, n_channels=len(ch),
                           channels="".join(ch), n_features=len(ch) * W, n_train_windows=int(tr.sum()), n_test_windows=int(te.sum()),
                           macro_f1=float(f1_score(yte, pred, labels=present, average="macro", zero_division=0)),
                           macro_f1_workouts=float(f1c[pw].mean()) if pw else np.nan,
                           accuracy=float(accuracy_score(yte, pred)),
                           weighted_f1=float(f1_score(yte, pred, labels=present, average="weighted", zero_division=0)),
                           macro_precision=float(precision_score(yte, pred, labels=present, average="macro", zero_division=0)),
                           macro_recall=float(recall_score(yte, pred, labels=present, average="macro", zero_division=0)),
                           train_time_s=t_fit, predict_time_batch_s=t_pred, final_train_loss=last_loss, epochs=TRAIN_PARAMS["epochs"])
                for i, l in enumerate(labels): row[f"f1_{l}"] = float(f1c[i]) if i in present else np.nan
                row.update(model_size_bytes=serialized_size(model), n_params=count_params(model), macs_per_window=count_macs(model, len(ch)),
                           data_rate_hz=len(ch) * FS)
                lf, lpd = measure_latency(model, pf["mu"], pf["sd"], ch_idx, te_rows)
                row.update(latency_feat_ms=lf, latency_predict_ms=lpd, latency_total_ms=lf + lpd)   # 'feat' = slice + z-score
                rows_buf.append(row)
                for i in range(K):
                    for j in range(K):
                        if cm[i, j]: conf_buf.append(dict(position=p, config=cname, model="CNN", fold=fd["fold"], seed=seed,
                                                          true=labels[i], pred=labels[j], n=int(cm[i, j])))
                done.add(key); n_new += 1
                del model
                if len(rows_buf) >= 10: flush()
                if n_new % 10 == 0 or n_new == 1:
                    el = time.time() - t_start; remaining = total_units - len(done)
                    log(f"[{len(done)}/{total_units}] {p} fold={fd['fold']} {cname} seed={seed}: macroF1={row['macro_f1']:.3f} "
                        f"loss={last_loss:.3f} fit={t_fit:.1f}s | elapsed {el/60:.1f} min, ETA ~ {el / n_new * remaining / 60:.1f} min")
        del pf, Xtr_c, Xte_c
        flush()
    del Xp_t
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
flush()
log("Loop finished. New units this session:", n_new, "| total done:", len(done), "of", total_units)

## 8 · Completeness check and descriptive summary
Descriptive only. Statistical comparisons belong in the analysis notebook. If several seeds were run, they are averaged within each fold first, so every fold counts once.

In [ ]:
res = pd.read_csv(RES_PATH)
exp = len(POS_LIST) * len(FOLD_LIST) * len(CONFIGS) * len(SEEDS)
log("Rows in results.csv:", len(res), "| expected for this grid:", exp)
dups = res.duplicated(subset=["position", "config", "model", "fold", "seed"]).sum()
assert dups == 0, f"{dups} duplicated units in results.csv"
if len(res) < exp: log("INCOMPLETE: re-run the notebook to finish the remaining units.")

num = ["macro_f1", "macro_f1_workouts", "accuracy", "weighted_f1", "n_channels", "n_params", "macs_per_window",
       "model_size_bytes", "latency_total_ms", "final_train_loss"]
fold_level = res.groupby(["position", "model", "config", "fold"])[num].mean().reset_index()
summ = (fold_level.groupby(["position", "model", "config"])
        .agg(n_folds=("fold", "count"), macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
             macro_f1_workouts_mean=("macro_f1_workouts", "mean"), accuracy_mean=("accuracy", "mean"),
             weighted_f1_mean=("weighted_f1", "mean"), n_channels=("n_channels", "first"),
             params=("n_params", "mean"), macs=("macs_per_window", "mean"), size_kb=("model_size_bytes", lambda x: x.mean() / 1024),
             latency_ms=("latency_total_ms", "mean"), final_train_loss=("final_train_loss", "mean"))
        .reset_index())
summ.to_csv(RUN_DIR / "summary_by_position_model_config.csv", index=False)
display(summ.round(4))

# descriptive side-by-side with the classical models (same folds, same windows), if notebook 01's full run is present
cl_path = CLASSICAL_ROOT / "full" / "summary_by_position_model_config.csv"
if cl_path.exists() and not SMOKE_TEST:
    cl = pd.read_csv(cl_path)
    both = pd.concat([cl[["position", "model", "config", "macro_f1_mean", "macro_f1_sd"]],
                      summ[["position", "model", "config", "macro_f1_mean", "macro_f1_sd"]]])
    cmp_ = both[both.config.isin(["6-axis", "acc-3", "gyro-3"])].pivot_table(index=["position", "config"], columns="model", values="macro_f1_mean")
    display(cmp_.round(3))
    cmp_.to_csv(RUN_DIR / "descriptive_vs_classical_macro_f1.csv")
else:
    log("Classical comparison skipped (smoke run, or", cl_path, "not found).")

## 9 · Environment record and zip

In [ ]:
import importlib
def ver(m):
    try: return importlib.import_module(m).__version__
    except Exception: return "n/a"
def cpu_name():
    try:
        for line in open("/proc/cpuinfo"):
            if line.lower().startswith("model name"): return line.split(":", 1)[1].strip()
    except Exception: pass
    return platform.processor() or "n/a"
env = {"python": sys.version, "platform": platform.platform(), "device": str(DEVICE),
       "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
       "cuda": torch.version.cuda, "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
       "cpu_for_latency": cpu_name(), "cpu_count": os.cpu_count(), "latency_threads": 1,
       "latency_note": "single window, batch 1, 1 CPU thread, slice+z-score+forward; hardware above; NOT edge-device latency",
       "seed_base": SEED, "seeds": SEEDS, "smoke_test": SMOKE_TEST,
       "packages": {m: ver(m) for m in ["numpy", "pandas", "sklearn", "torch"]},
       "data_sha256": DATA_SHA256, "windows_sha256": WINDOWS_SHA256, "config_hash": cfg_hash,
       "windows_verified_against_notebook_01": bool(p_parq.exists() or p_csv.exists()),
       "timestamp_utc": pd.Timestamp.now("UTC").isoformat()}
json.dump(env, open(RUN_DIR / "environment.json", "w"), indent=2)
(RUN_DIR / "run_log.txt").write_text("\n".join(LOG))

tag = "smoke" if SMOKE_TEST else "full"
zip_path = BASE / f"recgym_02_cnn_{tag}.zip"
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in RUN_DIR.rglob("*"):
        if f.is_file(): z.write(f, f"02_cnn_{tag}/{f.relative_to(RUN_DIR)}")
log("ZIP:", zip_path, "| MB:", round(os.path.getsize(zip_path) / 1e6, 2))
if SMOKE_TEST:
    log("Smoke run finished. Check the label-shuffle control, the loss values and the ETA above, then set SMOKE_TEST = False.")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    log("Not on Colab: find the zip in your working directory / Kaggle Output tab.")